In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Cleaning and Preprocessing


In [ ]:
import pandas as pd

df = pd.read_csv('train.csv')
df.head()
df.shape

(47199, 35)

In [ ]:
# drop useless columns
drop_cols = [
    "ID",
    "Total Received Interest",
    "Total Received Late Fee",
    "Recoveries",
    "Collection Recovery Fee",
    "Last week Pay",
    "Total Collection Amount"
]


df = df.drop(columns=drop_cols, errors="ignore")

# Handling Missing Values
num_cols = df.select_dtypes(include=["int64","float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[:0])


# Encode Categorical Values
categorical_cols = df.select_dtypes(include="object").columns

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

# Feature Engineeirng

df["Loan_to_Income"] = df["Loan Amount"] / df["Annual Income"]

df["Credit_Utilization"] = df["Revolving Balance"] / df["Total Revolving Credit Limit"]

df["High_DTI"] = (df["Debit to Income"] > df["Debit to Income"].median()).astype(int)


# normalisation
from sklearn.preprocessing import MinMaxScaler

credit_features = [
    "Delinquency - two years",
    "Inquires - six months",
    "Accounts Delinquent",
    "Revolving Utilities"
]

scaler = MinMaxScaler()

df_scaled = scaler.fit_transform(df[credit_features])
df_scaled = pd.DataFrame(df_scaled, columns=credit_features)


# Create CBIL_Score

df["CIBIL_Score"] = (
    850
    - (df_scaled["Delinquency - two years"] * 200)
    - (df_scaled["Inquires - six months"] * 150)
    - (df_scaled["Accounts Delinquent"] * 250)
    - (df_scaled["Revolving Utilities"] * 150)
)

df["CIBIL_Score"] = df["CIBIL_Score"].clip(300,850)


df.head()


,Loan Amount,Funded Amount,Funded Amount Investor,Term,Interest Rate,Annual Income,Debit to Income,Delinquency - two years,Inquires - six months,Open Account,...,Loan Title_personal,Loan Title_refi,Loan Title_relief,Loan Title_vacation,Initial List Status_w,Application Type_JOINT,Loan_to_Income,Credit_Utilization,High_DTI,CIBIL_Score
0,10000,32236,12329.36286,59,11.135007,176346.62670,16.284758,1,0,13,...,0,0,0,0,1,0,0.056707,3.663091,0,713.583688
1,3609,11940,12191.99692,59,12.237563,39833.92100,15.412409,0,0,12,...,0,0,0,0,0,0,0.090601,0.038880,0,733.580507
2,28276,9311,21603.22455,59,12.545884,91506.69105,28.137619,0,0,14,...,0,0,0,0,1,0,0.309005,0.070465,1,846.925100
3,11170,6954,17877.15585,59,16.731201,108286.57590,18.043730,1,0,7,...,0,0,0,0,1,0,0.103152,0.229498,0,724.683478
4,16890,13226,13539.92667,59,15.008300,44234.82545,17.209886,1,3,13,...,0,0,0,0,1,0,0.381826,0.068382,0,608.240606


In [ ]:
df.head()


,Loan Amount,Funded Amount,Funded Amount Investor,Term,Interest Rate,Annual Income,Debit to Income,Delinquency - two years,Inquires - six months,Open Account,...,Loan Title_personal,Loan Title_refi,Loan Title_relief,Loan Title_vacation,Initial List Status_w,Application Type_JOINT,Loan_to_Income,Credit_Utilization,High_DTI,CIBIL_Score
0,10000,32236,12329.36286,59,11.135007,176346.62670,16.284758,1,0,13,...,0,0,0,0,1,0,0.056707,3.663091,0,713.583688
1,3609,11940,12191.99692,59,12.237563,39833.92100,15.412409,0,0,12,...,0,0,0,0,0,0,0.090601,0.038880,0,733.580507
2,28276,9311,21603.22455,59,12.545884,91506.69105,28.137619,0,0,14,...,0,0,0,0,1,0,0.309005,0.070465,1,846.925100
3,11170,6954,17877.15585,59,16.731201,108286.57590,18.043730,1,0,7,...,0,0,0,0,1,0,0.103152,0.229498,0,724.683478
4,16890,13226,13539.92667,59,15.008300,44234.82545,17.209886,1,3,13,...,0,0,0,0,1,0,0.381826,0.068382,0,608.240606


In [ ]:
df.head()

,Loan Amount,Funded Amount,Funded Amount Investor,Term,Interest Rate,Annual Income,Debit to Income,Delinquency - two years,Inquires - six months,Open Account,...,Loan Title_personal,Loan Title_refi,Loan Title_relief,Loan Title_vacation,Initial List Status_w,Application Type_JOINT,Loan_to_Income,Credit_Utilization,High_DTI,CIBIL_Score
0,10000,32236,12329.36286,59,11.135007,176346.62670,16.284758,1,0,13,...,0,0,0,0,1,0,0.056707,3.663091,0,713.583688
1,3609,11940,12191.99692,59,12.237563,39833.92100,15.412409,0,0,12,...,0,0,0,0,0,0,0.090601,0.038880,0,733.580507
2,28276,9311,21603.22455,59,12.545884,91506.69105,28.137619,0,0,14,...,0,0,0,0,1,0,0.309005,0.070465,1,846.925100
3,11170,6954,17877.15585,59,16.731201,108286.57590,18.043730,1,0,7,...,0,0,0,0,1,0,0.103152,0.229498,0,724.683478
4,16890,13226,13539.92667,59,15.008300,44234.82545,17.209886,1,3,13,...,0,0,0,0,1,0,0.381826,0.068382,0,608.240606


# Model Training


In [ ]:
selected_features = [
    "Loan Amount",
    "Funded Amount",
    "Interest Rate",
    "Annual Income",
    "Debit to Income",
    "Delinquency - two years",
    "Inquires - six months",
    "Open Account",
    "Public Record",
    "Revolving Balance",
    "Revolving Utilities",
    "Total Accounts",
    "Total Current Balance",
    "Total Revolving Credit Limit",
    "Loan_to_Income",
    "Credit_Utilization",
    "CIBIL_Score",
    "High_DTI"
]
X = df[selected_features]
y = df["Loan Status"]

# y.value_counts()

,,,,,,,,,,,,,,,,,,count
Loan Amount,Funded Amount,Interest Rate,Annual Income,Debit to Income,Delinquency - two years,Inquires - six months,Open Account,Public Record,Revolving Balance,Revolving Utilities,Total Accounts,Total Current Balance,Total Revolving Credit Limit,Loan_to_Income,Credit_Utilization,CIBIL_Score,High_DTI,
35000,13852,16.169184,95893.78945,28.365724,0,0,8,0,2056,50.071827,25,30324,3436,0.364987,0.598370,775.551353,1,1
1014,20018,17.572383,41558.00529,23.850560,0,0,10,0,6582,42.902964,21,22744,15900,0.024400,0.413962,786.211384,1,1
1020,7684,7.546864,47793.29846,23.795058,0,0,20,0,16282,90.075078,6,68068,21524,0.021342,0.756458,716.066892,1,1
1024,10441,9.049422,206994.54290,13.767682,1,0,25,0,4452,39.109768,16,75111,29743,0.004947,0.149682,766.851832,0,1
1025,18208,7.877644,52062.13688,19.337442,0,0,12,0,4655,55.190981,15,72514,1057,0.019688,4.403974,767.939219,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1049,34581,7.728409,60984.67343,19.618388,0,0,14,0,9834,6.843665,26,30124,9672,0.017201,1.016749,839.831225,0,1
1048,31444,23.892889,47057.67737,20.531521,2,0,11,0,4335,92.494163,18,110387,43541,0.022271,0.099561,662.469735,0,1
1046,8235,5.776545,72431.53061,21.016249,1,1,10,0,7358,54.143470,25,127190,13335,0.014441,0.551781,714.496857,0,1


In [ ]:
# categorical_cols = X.select_dtypes(include='object').columns

# X = pd.get_dummies(
#     X,
#     columns=categorical_cols,
#     drop_first=True,
#     dtype=int
# )

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# constant_cols = [c for c in X_train.columns if X_train[c].nunique() <= 1]

# X_train = X_train.drop(columns=constant_cols)
# X_test = X_test.drop(columns=constant_cols)

# batch_cols = [c for c in X_train.columns if "Batch Enrolled" in c]

# X_train = X_train.drop(columns=batch_cols)
# X_test = X_test.drop(columns=batch_cols)

# loan_title_cols = [c for c in X_train.columns if "Loan Title" in c]

# X_train = X_train.drop(columns=loan_title_cols)
# X_test = X_test.drop(columns=loan_title_cols)

# y_train.value_counts()

,count
Loan Status,
0,48977
1,4993


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Baseline Model

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_scaled,y_train)




In [ ]:
y_train_res.value_counts()

,count
Loan Status,
0,48977
1,48977


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_res, y_train_res)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
y_pred

array([1, 1, 0, ..., 1, 1, 1])

In [ ]:
y_prob

array([0.51061854, 0.50098238, 0.48687117, ..., 0.54049088, 0.51294528,
       0.51158656])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

[[5752 6493]
 [ 569  679]]
              precision    recall  f1-score   support

           0       0.91      0.47      0.62     12245
           1       0.09      0.54      0.16      1248

    accuracy                           0.48     13493
   macro avg       0.50      0.51      0.39     13493
weighted avg       0.83      0.48      0.58     13493

ROC AUC: 0.5038892771513228


In [ ]:
#

# Random Forest

In [ ]:

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(X_train_res,y_train_res)

RandomForestClassifier(n_estimators=300, random_state=42)

In [ ]:
y_pred = rf.predict(X_test_scaled)
y_prob = rf.predict_proba(X_test_scaled)[:,1]

In [ ]:
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
y_prob

array([0.22666667, 0.19      , 0.12333333, ..., 0.24      , 0.29      ,
       0.18666667])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

[[12155    90]
 [ 1240     8]]
              precision    recall  f1-score   support

           0       0.91      0.99      0.95     12245
           1       0.08      0.01      0.01      1248

    accuracy                           0.90     13493
   macro avg       0.49      0.50      0.48     13493
weighted avg       0.83      0.90      0.86     13493

ROC AUC: 0.5036571703782811


# XGBoost


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1,
    random_state=42
)

xgb.fit(X_train_res,y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_prob = xgb.predict_proba(X_test_scaled)[:,1]

y_pred = (y_prob > 0.25).astype(int)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = xgb.predict(X_test_scaled)
y_prob = xgb.predict_proba(X_test_scaled)[:,1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

[[12239     6]
 [ 1248     0]]
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     12245
           1       0.00      0.00      0.00      1248

    accuracy                           0.91     13493
   macro avg       0.45      0.50      0.48     13493
weighted avg       0.82      0.91      0.86     13493

ROC AUC: 0.5143931392719163


In [ ]:
X_train_res.shape

(97954, 18)

In [ ]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False))

Open Account                    0.203217
Total Accounts                  0.064259
Delinquency - two years         0.061480
Debit to Income                 0.059438
Total Current Balance           0.058382
Funded Amount                   0.058190
Interest Rate                   0.056472
Total Revolving Credit Limit    0.055439
Revolving Balance               0.055006
Loan Amount                     0.052838
Annual Income                   0.052581
Credit_Utilization              0.051887
Loan_to_Income                  0.050778
Revolving Utilities             0.049970
CIBIL_Score                     0.049821
Inquires - six months           0.008090
High_DTI                        0.006156
Public Record                   0.005997
dtype: float64


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42
)

lgbm.fit(X_train_res, y_train_res)

[LightGBM] [Info] Number of positive: 48977, number of negative: 48977
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4588
[LightGBM] [Info] Number of data points in the train set: 97954, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=500,
               random_state=42)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[[12245     0]
 [ 1248     0]]
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     12245
           1       0.00      0.00      0.00      1248

    accuracy                           0.91     13493
   macro avg       0.45      0.50      0.48     13493
weighted avg       0.82      0.91      0.86     13493

ROC AUC: 0.5140691255457487


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,roc_auc_score
import pandas as pd

models = {
    "Logistic Regression": model,
    "Random Forest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm
}

results = []

for name, model in models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)


    results.append({
        "Model": name,
        "TN": cm[0,0],
        "FP": cm[0,1],
        "FN": cm[1,0],
        "TP": cm[1,1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    })

results_df = pd.DataFrame(results)

print(results_df)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


                 Model    TN     FP   FN    TP  Accuracy  Precision    Recall  \
0  Logistic Regression    72  12173   14  1234  0.096791   0.092041  0.988782   
1        Random Forest  2803   9442  306   942  0.277551   0.090716  0.754808   
2              XGBoost   133  12112   19  1229  0.100941   0.092122  0.984776   
3             LightGBM    72  12173   14  1234  0.096791   0.092041  0.988782   

   F1 Score   ROC AUC  
0  0.168407  0.497331  
1  0.161967  0.491859  
2  0.168483  0.497819  
3  0.168407  0.497331  
